*This will be the main notebook for practicing git*

# Git Solo Workflow — Reference Notes

---

## The Three Zones

Every file lives in one of three zones. Knowing which zone tells you what command to use.

**Zone 1 — Working Directory**
Files on disk. Git can see changes here but is not tracking them yet.

↓ `git add <file>`

**Zone 2 — Staging Area (Index)**
A holding area where you choose exactly what goes into the next commit.
You can stage some files and leave others out.

↓ `git commit -m "message"`

**Zone 3 — Repository (.git folder)**
Permanent history. Each commit gets a unique hash ID you can refer back to.

↓ `git push`

**Zone 4 — Remote Repository (GitHub)**
Your repository hosted online. Commits only arrive here when you explicitly push them.
Nothing you do locally (including `git commit`) sends changes here automatically.

> The staging area exists so you can edit many files but commit only the ones that are ready.
> Clean, focused commits are a professional habit.

---

## Core Commands

| Command | What it does |
|---|---|
| `git init` | Create a new repository in the current folder |
| `git status` | Show which zone each changed file is in |
| `git add <file>` | Move a file from working directory → staging |
| `git commit -m "msg"` | Save staged changes as a permanent snapshot |
| `git log --oneline` | Show commit history, one line per commit |
| `git diff <file>` | Show line-by-line changes vs the last commit |

---

## Undoing Things

The right undo command depends on which zone the change is in.

### Zone 1 — Change is on disk, not yet staged
```
git restore <file>
```
⚠ **Destructive.** The change is gone permanently — git never knew about it.

### Zone 2 — Change is staged, not yet committed
```
git restore --staged <file>
```
✓ **Safe.** The change moves back to the working directory, nothing is lost.

### Zone 3 — Change is already committed

**Safe option — `git revert`**
```
git revert <hash>
```
Adds a new commit that cancels the target commit. History is preserved.
Use this on any shared or remote branch.

**Nuclear option — `git reset`**
```
git reset --soft <hash>   # uncommit, keep changes staged
git reset --mixed <hash>  # uncommit, unstage, keep files on disk (default)
git reset --hard <hash>   # uncommit and delete changes from disk
```
⚠ `--hard` is irreversible. Never use reset on a branch others are working on.

---

## Commit Message Convention

Format: `<type>: <short description>`

A good message completes: *"If applied, this commit will..."*

| Type | When to use |
|---|---|
| `feat` | New feature |
| `fix` | Bug fix |
| `docs` | Documentation only |
| `refactor` | Restructuring without behaviour change |
| `chore` | Housekeeping — config, dependencies, gitignore |

---

## `git status` and `git log` — Your Navigation Tools

> These two commands should be instinctive. Run them constantly. Neither changes anything — always safe.

**`git status`**
Shows where you are right now — what is modified, staged, or untracked.
Also prints hints about what to run next. Read the full output, not just the filenames.

**`git log --oneline`**
Shows where you have been — full commit history with short hashes.
Those hashes are how you refer to commits in `git diff`, `git revert`, and `git reset`.


# ML Project Structure

---

## Standard Layout

```
my_project/
│
├── README.md              # What the project is and how to run it
├── requirements.txt       # pip dependencies
├── .gitignore             # What git should never track
│
├── data/
│   ├── raw/               # Original data, never modified
│   └── processed/         # Cleaned and ready to train on
│
├── notebooks/             # Exploration only, not imported elsewhere
├── src/                   # Importable Python package
│   ├── __init__.py
│   ├── model.py
│   ├── train.py
│   └── utils.py
│
├── checkpoints/           # Saved model weights — gitignored
└── outputs/               # Plots, logs, results
```

**`src/`** is a proper Python package because it contains `__init__.py`.
This means you can do `from src.model import GPT` anywhere in the project.

**`data/` and `checkpoints/`** are gitignored — datasets and weights are too large for git
and belong in cloud storage or a model registry (HuggingFace Hub, MLflow, W&B).

---

## `.gitignore` for ML Projects

```
data/
checkpoints/
*.pt
*.pth
*.h5
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
venv/
.DS_Store
Thumbs.db
```

---

## Keeping API Keys Secret

**Step 1** — store keys in a `.env` file:
```
OPENAI_API_KEY=sk-abc123...
```

**Step 2** — add `.env` to `.gitignore` before your first commit.

**Step 3** — load in Python:
```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
```

**Step 4** — commit a `.env.example` with placeholder values so collaborators know what's needed:
```
OPENAI_API_KEY=your_key_here
```

> ⚠ If you accidentally commit a secret, assume it is compromised — revoke it immediately.
> Deleting the file in a new commit is not enough; the secret remains in git history.
